# Crear archivo HDF5 con todas las imágenes NIH

Este notebook convierte los ~25,000 PNGs sueltos en un solo archivo `.h5`.

**Se ejecuta UNA SOLA VEZ.** Después de esto, cada sesión de entrenamiento
solo copia 1 archivo (~2 GB) en vez de 25,000 archivos.

**Tiempo:** ~15-20 min.
**Espacio:** ~2-3 GB adicionales en Drive (los PNGs los puedes borrar después).
**Runtime:** CPU (no necesita GPU).


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
IMAGES_DIR   = f'{NIH_DIR}/images'
H5_PATH      = f'{NIH_DIR}/nih_images.h5'

import os
print(f'Drive montado correctamente')
print(f'H5 destino: {H5_PATH}')


## 2. Instalar h5py

In [ ]:
!pip install -q h5py
import h5py
print(f'h5py version: {h5py.__version__}')


## 3. Verificar imágenes disponibles

In [ ]:
import pandas as pd

# Leer CSV — ya sabemos qué imágenes necesitamos
df = pd.read_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv')
needed_images = df['Image Index'].tolist()
print(f'Imágenes en el CSV: {len(needed_images):,}')
print(f'Usaremos todas las del CSV directamente.')
print(f'Si alguna no existe en Drive, se registra como fallida.')

to_include = needed_images  # iterar desde CSV, verificar al abrir

## 4. Crear el archivo HDF5

In [ ]:
import h5py

try:
    hf = h5py.File(H5_PATH, 'r', swmr=True)
    print(hf.keys())
except Exception as e:
    print(e)

In [ ]:
with h5py.File(H5_PATH, 'r', swmr=True) as hf:
    print("Grupos:", list(hf.keys()))
    print("Imágenes guardadas:", len(hf['images'].keys()))

In [ ]:
with h5py.File(H5_PATH, 'r', swmr=True) as hf:
    already_done = set(hf['images'].keys())

In [ ]:
import h5py

with h5py.File(H5_PATH, 'a') as hf:
    print("HDF5 abierto correctamente en modo append")

In [ ]:
import h5py
import numpy as np
from PIL import Image
from tqdm import tqdm
import time

if os.path.exists(H5_PATH):
    size_gb = os.path.getsize(H5_PATH) / (1024**3)
    print(f'✓ Ya existe {H5_PATH} ({size_gb:.2f} GB)')
else:
    images_list = to_include  # lista desde CSV
    n = len(images_list)
    print(f'Creando HDF5 con hasta {n:,} imágenes...')
    t0 = time.time()

    failed = []
    saved  = 0

    with h5py.File(H5_PATH, 'w') as hf:
        grp = hf.create_group('images')

        for img_name in tqdm(images_list, desc='Guardando'):
            img_path = f'{IMAGES_DIR}/{img_name}'
            try:
                with Image.open(img_path) as pil_img:
                    pil_img = pil_img.convert('L')
                    # Resize a 256x256 para reducir tamaño del HDF5
                    pil_resized = pil_img.resize((256, 256), Image.LANCZOS)
                    arr = np.array(pil_resized, dtype=np.uint8)
                grp.create_dataset(img_name, data=arr,
                                   compression='gzip', compression_opts=4)
                saved += 1
            except OSError as e:
                failed.append(img_name)
                # No detener el proceso por un archivo con error
                continue

        # Guardar nombres de las imágenes que sí quedaron
        saved_names = [n for n in images_list if n not in set(failed)]
        dt = h5py.special_dtype(vlen=str)
        hf.create_dataset('image_names',
                          data=np.array(saved_names, dtype=object), dtype=dt)

    elapsed = time.time() - t0
    size_gb = os.path.getsize(H5_PATH) / (1024**3)
    print(f'\n✓ HDF5 creado en {elapsed/60:.1f} min')
    print(f'  Tamaño:   {size_gb:.2f} GB')
    print(f'  Guardadas: {saved:,} / {n:,}')
    print(f'  Fallidas:  {len(failed):,}')
    if failed[:5]:
        print(f'  Primeras fallidas: {failed[:5]}')


## 5. Verificar el HDF5

In [ ]:
import h5py

with h5py.File(H5_PATH, 'r') as hf:
    n_names = len(hf['image_names'])
    n_images = len(hf['images'])
    # Leer una imagen de prueba
    first_name = hf['image_names'][0].decode() if isinstance(hf['image_names'][0], bytes) else hf['image_names'][0]
    sample = hf['images'][first_name][:]

print(f'Imágenes en HDF5: {n_images:,}')
print(f'Primera imagen: {first_name}')
print(f'Shape: {sample.shape}, dtype: {sample.dtype}')
print(f'Range: [{sample.min()}, {sample.max()}]')
print()
print('✓ HDF5 válido y listo para usar.')
print(f'  Guarda esta ruta: {H5_PATH}')


In [ ]:
import h5py, pandas as pd

H5_PATH = f'{NIH_DIR}/nih_images.h5'

# Qué imágenes están en el HDF5
with h5py.File(H5_PATH, 'r') as hf:
    in_h5 = set(k for k in hf['images'].keys())

print(f'Imágenes en HDF5: {len(in_h5):,}')

# Cuántas de nuestros splits están en el HDF5
processed_s2 = f'{PROJECT_ROOT}/data/processed_s2'
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed_s2}/{split}.csv')
    in_split = set(df['Image Index'])
    overlap = in_split & in_h5
    pct = len(overlap) / len(in_split) * 100
    print(f'{split:5s}: {len(overlap)}/{len(in_split)} en HDF5 ({pct:.0f}%)')
    for label in range(4):
        df_label = df[df['label'] == label]
        n_ok = len(set(df_label['Image Index']) & in_h5)
        print(f'  clase {label}: {n_ok}/{len(df_label)} imágenes disponibles')

In [ ]:
import h5py, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm
import time, os

# Qué falta agregar
with h5py.File(H5_PATH, 'r') as hf:
    already_done = set(hf['images'].keys())

all_needed = set(pd.read_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv')['Image Index'])
pending = list(all_needed - already_done)
print(f'En HDF5: {len(already_done):,} | Pendientes: {len(pending):,}')

if not pending:
    print('✓ HDF5 completo')
else:
    failed = []
    saved  = 0
    t0 = time.time()

    # Abrir en modo append para agregar sin borrar lo que ya hay
    with h5py.File(H5_PATH, 'a') as hf:
        grp = hf['images']
        for img_name in tqdm(pending, desc='Completando HDF5'):
            try:
                with Image.open(f'{IMAGES_DIR}/{img_name}') as pil_img:
                    pil_resized = pil_img.convert('L').resize((256, 256), Image.LANCZOS)
                    arr = np.array(pil_resized, dtype=np.uint8)
                grp.create_dataset(img_name, data=arr,
                                   compression='gzip', compression_opts=4)
                saved += 1
            except OSError:
                failed.append(img_name)

    elapsed = time.time() - t0
    total_now = len(already_done) + saved
    print(f'\n✓ Ronda completada en {elapsed/60:.1f} min')
    print(f'  Agregadas: {saved:,} | Fallidas esta ronda: {len(failed):,}')
    print(f'  Total en HDF5: {total_now:,} / {len(all_needed):,}')
    print()
    if failed:
        print('Vuelve a correr esta celda para reintentar las fallidas.')
    else:
        print('✓ HDF5 completo.')

In [ ]:
import h5py, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
import numpy as np

with h5py.File(H5_PATH, 'r') as hf:
    in_h5 = set(hf['images'].keys())

print(f'Imágenes en HDF5: {len(in_h5):,}')

# Filtrar el CSV de splits a solo imágenes disponibles en HDF5
processed_s2 = f'{PROJECT_ROOT}/data/processed_s2'

for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed_s2}/{split}.csv')
    df_filtered = df[df['Image Index'].isin(in_h5)].reset_index(drop=True)
    df_filtered.to_csv(f'{processed_s2}/{split}.csv', index=False)
    print(f'{split}: {len(df)} → {len(df_filtered)} imgs')
    for label in range(4):
        n = (df_filtered['label'] == label).sum()
        print(f'  clase {label}: {n}')

print('\n✓ Splits actualizados para usar solo imágenes en HDF5')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
IMAGES_DIR   = f'{NIH_DIR}/images'

H5_LOCAL = '/content/nih_images.h5'          # creamos aquí (rápido)
H5_DRIVE = f'{NIH_DIR}/nih_images.h5'        # copiamos aquí al final

import os, h5py, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm
import time, shutil

!pip install -q h5py
print('Setup OK')

In [ ]:
df = pd.read_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv')
images_list = df['Image Index'].unique().tolist()
print(f'Imágenes a procesar: {len(images_list):,}')

In [ ]:
import os

print("Existe en local:", os.path.exists(H5_LOCAL))
print("Existe en Drive:", os.path.exists(H5_DRIVE))

if os.path.exists(H5_DRIVE):
    print("Tamaño Drive GB:", os.path.getsize(H5_DRIVE)/(1024**3))

In [ ]:
import h5py

with h5py.File(H5_DRIVE, 'r', swmr=True) as hf:
    print("Imágenes:", len(hf['images']))

## 6. Siguiente paso

El archivo HDF5 está listo en Drive. Ahora cada sesión de entrenamiento:

1. Copia `nih_images.h5` a `/content/` (~2-3 min en vez de 25k archivos).
2. El Dataset lee del HDF5 local.
3. Sin errores OSError, sin timeouts de Drive.

Vuelve al chat para recibir el Dataset actualizado que lee HDF5.
